# Publish model cards + weights to the HF Hub

Renders each model's card from `model_cards_templates/` — scores, size and latency from
`notebooks/published_models_corrected_eval.csv`, hyperparameters from `mlflow.db` — writes the
filled cards to `model-cards/`, exports the Lightning checkpoint's backbone weights as safetensors,
and pushes both to the Hub.

The push is the **last cell** and is the only step that touches the network. Run the cells above it
first and read the generated cards in `model-cards/` before you push.

In [1]:
import sqlite3
import string
from pathlib import Path

import pandas as pd
import torch
from huggingface_hub import HfApi, ModelCard
from safetensors.torch import save_file

BASE_DIR = Path.cwd().parent
CHECKPOINTS = BASE_DIR / "models_checkpoints"
TEMPLATES = BASE_DIR / "model_cards_templates"
MODEL_CARDS = BASE_DIR / "model-cards"
MODEL_WEIGHTS = BASE_DIR / "model-weights"
EVAL_CSV = BASE_DIR / "notebooks" / "published_models_corrected_eval.csv"
DB_PATH = BASE_DIR / "mlflow.db"
MODEL_WEIGHTS.mkdir(exist_ok=True)

# `key` is the prefix other cards use to reference this model's numbers ($vit_test_f1 etc).
# Checkpoints are found by run_id glob and cards by `{repo_id}-readme.md`, so no filename
# is written down twice.
MODELS = [
    {
        "key": "convnext",
        "run_id": "01d3e87a",
        "repo_id": "convnext-tiny-flower-classifier",
        "display": "ConvNeXt-Tiny",
        "best_for": "default serving choice",
    },
    {
        "key": "vit",
        "run_id": "620728ca",
        "repo_id": "vit-flower-classifier",
        "display": "ViT-B/16",
        "best_for": "maximum macro-F1",
    },
    {
        "key": "effnet",
        "run_id": "a6094159",
        "repo_id": "efficientnetv2-s-flower-classifier",
        "display": "EfficientNetV2-S",
        "best_for": "smallest weights",
    },
]

# Leak-era counterpart of each model, re-scored on the corrected split. Cards quote these to say
# what the retrain actually bought ($vit_leak_test_f1 etc).
LEAK_ERA_RUN_IDS = {"convnext": "4632c1bb", "vit": "6899e8e2", "effnet": "d6508546"}

evals = pd.read_csv(EVAL_CSV, dtype={"run_id": str}).set_index("run_id")
evals.loc[[m["run_id"] for m in MODELS]]

,architecture,experiment,val_loss,val_acc,val_f1,test_loss,test_acc,test_f1,num_parameters,model_size_mb,checkpoint_size_mb,latency_ms_mean,latency_ms_p95
run_id,,,,,,,,,,,,,
01d3e87a,convnext_tiny,corrected-split-retrain,0.062288,0.986971,0.979339,0.076303,0.982085,0.969399,27898566,111.594672,335.018055,3.522369,4.467439
620728ca,vit_b_16,corrected-split-retrain,0.079657,0.983713,0.973708,0.076175,0.983713,0.972163,85877094,343.508784,1030.723325,5.385118,5.676090
a6094159,efficientnet_v2_s,corrected-split-retrain,0.126331,0.969870,0.949827,0.110990,0.969055,0.943287,20308150,81.849376,245.016185,11.334295,12.840330


## Pull the numbers

Scores and cost from the eval CSV, hyperparameters from the MLflow run that produced the checkpoint.
Nothing about a model is typed into this notebook by hand — if a number is wrong here, it is wrong at
the source.

In [2]:
OPTIMIZER_DISPLAY = {"adamw": "AdamW", "adam": "Adam", "sgd": "SGD"}


def hyperparameters(run_id: str) -> dict[str, str]:
    """Card-facing hyperparameter strings for a run, straight from mlflow.db."""
    conn = sqlite3.connect(DB_PATH)
    try:
        p = dict(
            conn.execute(
                "SELECT p.key, p.value FROM params p JOIN runs r ON r.run_uuid = p.run_uuid "
                "WHERE r.run_uuid LIKE ?",
                (f"{run_id}%",),
            ).fetchall()
        )
    finally:
        conn.close()
    if not p:
        raise ValueError(f"no params logged for run {run_id!r} in {DB_PATH}")

    # ponytail: only cosine gets its kwargs spelled out, since that is all three runs. Any other
    # scheduler renders as its bare name — add a branch here if a card ever needs one.
    scheduler = p["scheduler"]
    if scheduler == "cosine":
        scheduler = (
            f"Cosine annealing (T_max={p['scheduler_kwargs/T_max']}, "
            f"eta_min={p['scheduler_kwargs/eta_min']})"
        )

    return {
        "optimizer": OPTIMIZER_DISPLAY.get(p["optimizer"], p["optimizer"]),
        "scheduler": scheduler,
        **{
            k: p[k]
            for k in (
                "lr_head_stage_1",
                "lr_head_stage_2",
                "lr_backbone",
                "unfreeze_at_epoch",
                "max_epochs",
                "batch_size",
                "effective_batch_size",
                "accumulate_grad_batches",
                "precision",
                "weight_decay",
                "early_stopping_patience",
            )
        },
    }


def measurements(run_id: str) -> dict[str, str]:
    """Card-facing score/size/latency strings for a run, straight from the eval CSV."""
    r = evals.loc[run_id]
    return {
        "val_acc": f"{r.val_acc:.4f}",
        "val_f1": f"{r.val_f1:.4f}",
        "test_acc": f"{r.test_acc:.4f}",
        "test_f1": f"{r.test_f1:.4f}",
        "val_loss": f"{r.val_loss:.4f}",
        "test_loss": f"{r.test_loss:.4f}",
        "num_parameters": f"{int(r.num_parameters):,}",
        "model_size_mb": f"{r.model_size_mb:.1f}",
        "checkpoint_size_mb": f"{r.checkpoint_size_mb:.1f}",
        "latency_ms_mean": f"{r.latency_ms_mean:.2f}",
        "latency_ms_p95": f"{r.latency_ms_p95:.2f}",
    }


for m in MODELS:
    m["numbers"] = measurements(m["run_id"]) | hyperparameters(m["run_id"])

pd.DataFrame({m["key"]: m["numbers"] for m in MODELS})

,convnext,vit,effnet
val_acc,0.9870,0.9837,0.9699
val_f1,0.9793,0.9737,0.9498
test_acc,0.9821,0.9837,0.9691
test_f1,0.9694,0.9722,0.9433
val_loss,0.0623,0.0797,0.1263
test_loss,0.0763,0.0762,0.1110
num_parameters,"27,898,566","85,877,094","20,308,150"
model_size_mb,111.6,343.5,81.8
checkpoint_size_mb,335.0,1030.7,245.0
latency_ms_mean,3.52,5.39,11.33


## Comparison table

One block shared by all three cards, with the current model's row bolded. The two v1 baselines
predate the Lightning pipeline and have no comparable F1, hence the dashes.

In [3]:
BASELINE_ROWS = [
    "| [SimpleCNN (scratch)](https://huggingface.co/bengid/flower-classifier/blob/main/flower_model_weights.pth) | ~0.63 | - | - | - | historical baseline only |",
    "| [EfficientNet-B0 (v1, partial unfreeze)](https://huggingface.co/bengid/flower-classifier/blob/main/ft_EfficientNet-B0.pth) | >0.93 | - | - | - | historical baseline only |",
]


def comparison_table(current_key: str) -> str:
    """Markdown table of all published models, current one bolded, best test_f1 first."""
    ranked = sorted(MODELS, key=lambda m: float(m["numbers"]["test_f1"]), reverse=True)

    rows = list(BASELINE_ROWS)
    for m in ranked:
        n = m["numbers"]
        cells = [n["test_acc"], n["test_f1"], n["num_parameters"], n["model_size_mb"]]
        link = f"[{m['display']}](https://huggingface.co/bengid/{m['repo_id']})"
        if m["key"] == current_key:
            name = f"**{link} (this model)**"
            cells = [f"**{c}**" for c in cells]
        else:
            name = link
        rows.append(f"| {name} | " + " | ".join(cells) + f" | {m['best_for']} |")

    header = [
        "| Model | Test Acc | Test F1 | Params | Size (MB) | Best For |",
        "|---|---|---|---|---|---|",
    ]
    return "\n".join(header + rows)


print(comparison_table("convnext"))

| Model | Test Acc | Test F1 | Params | Size (MB) | Best For |
|---|---|---|---|---|---|
| [SimpleCNN (scratch)](https://huggingface.co/bengid/flower-classifier/blob/main/flower_model_weights.pth) | ~0.63 | - | - | - | historical baseline only |
| [EfficientNet-B0 (v1, partial unfreeze)](https://huggingface.co/bengid/flower-classifier/blob/main/ft_EfficientNet-B0.pth) | >0.93 | - | - | - | historical baseline only |
| [ViT-B/16](https://huggingface.co/bengid/vit-flower-classifier) | 0.9837 | 0.9722 | 85,877,094 | 343.5 | maximum macro-F1 |
| **[ConvNeXt-Tiny](https://huggingface.co/bengid/convnext-tiny-flower-classifier) (this model)** | **0.9821** | **0.9694** | **27,898,566** | **111.6** | default serving choice |
| [EfficientNetV2-S](https://huggingface.co/bengid/efficientnetv2-s-flower-classifier) | 0.9691 | 0.9433 | 20,308,150 | 81.8 | smallest weights |


## Render the cards

In [4]:
# Every model's numbers are exposed to every card under its key prefix, so a card can quote a
# sibling model ($vit_test_f1, $convnext_model_size_mb, ...) without anything being hardcoded.
cross = {
    f"{m['key']}_{field}": value
    for m in MODELS
    for field, value in m["numbers"].items()
} | {
    f"{key}_leak_{field}": value
    for key, run_id in LEAK_ERA_RUN_IDS.items()
    for field, value in measurements(run_id).items()
}

for m in MODELS:
    card = f"{m['repo_id']}-readme.md"
    template = string.Template((TEMPLATES / card).read_text())
    # .substitute (not .safe_substitute): an unknown placeholder must fail here, not ship to the Hub
    rendered = template.substitute(
        cross | m["numbers"] | {"comparison_table": comparison_table(m["key"])}
    )
    assert "$" not in rendered, f"unsubstituted placeholder left in {card}"

    (MODEL_CARDS / card).write_text(rendered)
    print(f"wrote {MODEL_CARDS / card} ({len(rendered):,} chars)")

wrote /home/zelluzy/Desktop/code/flowers/model-cards/convnext-tiny-flower-classifier-readme.md (12,763 chars)
wrote /home/zelluzy/Desktop/code/flowers/model-cards/vit-flower-classifier-readme.md (13,437 chars)
wrote /home/zelluzy/Desktop/code/flowers/model-cards/efficientnetv2-s-flower-classifier-readme.md (14,139 chars)


## Export weights as safetensors

The Lightning checkpoint wraps the torchvision backbone as `self.model` and also carries optimizer
state and a `criterion.weight` buffer. Only the `model.*` tensors are published, with the prefix
stripped, so the file loads straight into a bare torchvision model (see each card's Usage section).

In [5]:
def export_safetensors(run_id: str, out_path: Path) -> Path:
    ckpts = list(CHECKPOINTS.glob(f"*-{run_id}.ckpt"))
    if len(ckpts) != 1:
        raise FileNotFoundError(f"expected exactly 1 checkpoint for {run_id}, found {ckpts}")

    ckpt = torch.load(ckpts[0], map_location="cpu", weights_only=True)
    state_dict = {
        k.removeprefix("model."): v.contiguous()
        for k, v in ckpt["state_dict"].items()
        if k.startswith("model.")
    }
    save_file(state_dict, out_path)
    return out_path


for m in MODELS:
    path = export_safetensors(m["run_id"], MODEL_WEIGHTS / f"{m['repo_id']}.safetensors")
    print(f"{path.name}: {path.stat().st_size / 1e6:.1f} MB")

convnext-tiny-flower-classifier.safetensors: 111.6 MB
vit-flower-classifier.safetensors: 343.5 MB
efficientnetv2-s-flower-classifier.safetensors: 81.9 MB


## Push to the Hub

Network step. Requires `huggingface-cli login` (or `HF_TOKEN`). Running this overwrites the live
cards and weights for all three repos — check `model-cards/` first.

In [6]:
api = HfApi()

for m in MODELS:
    repo_id = f"bengid/{m['repo_id']}"
    api.create_repo(repo_id, exist_ok=True)
    api.upload_file(
        path_or_fileobj=MODEL_WEIGHTS / f"{m['repo_id']}.safetensors",
        path_in_repo=f"{m['repo_id']}.safetensors",
        repo_id=repo_id,
    )
    card = MODEL_CARDS / f"{m['repo_id']}-readme.md"
    ModelCard(content=card.read_text()).push_to_hub(repo_id)
    print(f"pushed {repo_id}")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

pushed bengid/convnext-tiny-flower-classifier


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

pushed bengid/vit-flower-classifier


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

pushed bengid/efficientnetv2-s-flower-classifier
